# Chapter 1 Baseline Model Readiness Check

This notebook performs the pre-implementation readiness check for Sprint 2 / Issue 2: ASIC Chapter 1 baseline risk models.

It inspects the frozen Chapter 1 preprocessing outputs already present in this repository and checks whether the required preconditions are met before logistic regression and XGBoost are implemented.

This notebook does **not** train any models.


In [1]:
from __future__ import annotations

import json
import re
import sys
from pathlib import Path

import pandas as pd

try:
    from IPython.display import Markdown, display
except Exception:
    def Markdown(text: str) -> str:
        return text

    def display(obj: object) -> None:
        print(obj)


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (
            candidate / "src" / "chapter1_mortality_decomposition"
        ).exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root. Open the notebook from inside "
        "the 1-mortality-decomposition repository."
    )


PROJECT_ROOT = find_project_root()
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "chapter1"
SRC_ROOT = PROJECT_ROOT / "src"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 200)

display(
    Markdown(
        f"""
**Project root:** `{PROJECT_ROOT}`

**Artifact root:** `{ARTIFACT_ROOT}`

This notebook inspects frozen Chapter 1 preprocessing outputs only. It does **not** train any models.
"""
    )
)



**Project root:** `/Users/joanameyer/repository/1-mortality-decomposition`

**Artifact root:** `/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1`

This notebook inspects frozen Chapter 1 preprocessing outputs only. It does **not** train any models.


In [2]:
REQUIRED_HORIZONS = [8, 16, 24, 48, 72]
PRIMARY_HORIZON = 24
SECONDARY_HORIZON = 48
SENSITIVITY_HORIZONS = [8, 16, 72]
TEXT_SUFFIXES = {".py", ".md", ".toml", ".json", ".sh"}

artifact_paths = {
    "retained_stay_table": "artifacts/chapter1/cohort/chapter1_retained_stay_table.csv",
    "retained_hospitals": "artifacts/chapter1/cohort/chapter1_retained_hospitals.csv",
    "site_counts_summary": "artifacts/chapter1/cohort/chapter1_site_counts_summary.csv",
    "cohort_notes": "artifacts/chapter1/cohort/chapter1_notes.csv",
    "cohort_verification_summary": "artifacts/chapter1/cohort/chapter1_verification_summary.csv",
    "candidate_instances": "artifacts/chapter1/instances/chapter1_candidate_instances.csv",
    "valid_instances": "artifacts/chapter1/instances/chapter1_valid_instances.csv",
    "proxy_labels": "artifacts/chapter1/labels/chapter1_proxy_horizon_labels.csv",
    "usable_proxy_labels": "artifacts/chapter1/labels/chapter1_usable_proxy_horizon_labels.csv",
    "proxy_label_summary_by_horizon": "artifacts/chapter1/labels/chapter1_proxy_label_summary_by_horizon.csv",
    "proxy_unlabeled_reason_summary": "artifacts/chapter1/labels/chapter1_proxy_unlabeled_reason_summary.csv",
    "feature_set_definition": "artifacts/chapter1/feature_sets/chapter1_feature_set_definition.csv",
    "feature_set_validation_summary": "artifacts/chapter1/feature_sets/chapter1_feature_set_validation_summary.csv",
    "primary_model_ready": "artifacts/chapter1/model_ready/chapter1_primary_model_ready_dataset.csv",
    "extended_model_ready": "artifacts/chapter1/model_ready/chapter1_extended_model_ready_dataset.csv",
    "primary_readiness_summary": "artifacts/chapter1/model_ready/chapter1_primary_readiness_summary.csv",
    "primary_carry_forward_verification": "artifacts/chapter1/carry_forward/chapter1_primary_carry_forward_verification_summary.csv",
    "primary_missingness_by_hospital_and_family": "artifacts/chapter1/carry_forward/chapter1_primary_missingness_by_hospital_and_family.csv",
    "stay_split_assignments": "artifacts/chapter1/splits/chapter1_stay_split_assignments.csv",
    "stay_split_summary": "artifacts/chapter1/splits/chapter1_stay_split_summary.csv",
    "stay_split_verification_summary": "artifacts/chapter1/splits/chapter1_stay_split_verification_summary.csv",
    "primary_split_summary": "artifacts/chapter1/splits/chapter1_primary_split_summary.csv",
    "primary_split_verification_summary": "artifacts/chapter1/splits/chapter1_primary_split_verification_summary.csv",
}


def load_csv(relative_path: str) -> pd.DataFrame | None:
    path = PROJECT_ROOT / relative_path
    if not path.exists():
        return None
    return pd.read_csv(path)


def read_text(relative_path: str) -> str:
    return (PROJECT_ROOT / relative_path).read_text()


def has_columns(df: pd.DataFrame | None, columns: list[str]) -> bool:
    return df is not None and set(columns).issubset(df.columns)


def as_bool_series(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    normalized = series.astype(str).str.strip().str.lower()
    return normalized.map({"true": True, "false": False, "1": True, "0": False}).fillna(False)


def as_bool_value(value: object) -> bool | None:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    if isinstance(value, bool):
        return value
    normalized = str(value).strip().lower()
    if normalized in {"true", "1"}:
        return True
    if normalized in {"false", "0"}:
        return False
    return None


def metric_value(df: pd.DataFrame | None, metric_name: str) -> object:
    if df is None or df.empty or "metric" not in df.columns:
        return None
    matches = df.loc[df["metric"].eq(metric_name), "value"]
    if matches.empty:
        return None
    value = matches.iloc[0]
    return None if pd.isna(value) else value


def check_value(df: pd.DataFrame | None, check_id: str) -> bool | None:
    if df is None or df.empty or "check_id" not in df.columns:
        return None
    matches = df.loc[df["check_id"].eq(check_id), "passed"]
    if matches.empty:
        return None
    return as_bool_value(matches.iloc[0])


def search_repo(pattern: str, targets: list[str]) -> list[str]:
    regex = re.compile(pattern, flags=re.IGNORECASE)
    hits: list[str] = []
    for target in targets:
        path = PROJECT_ROOT / target
        if not path.exists():
            continue
        if path.is_dir():
            files = [p for p in path.rglob("*") if p.is_file() and p.suffix in TEXT_SUFFIXES]
        else:
            files = [path]
        for file_path in files:
            text = file_path.read_text(errors="ignore")
            for lineno, line in enumerate(text.splitlines(), start=1):
                if regex.search(line):
                    hits.append(f"{file_path.relative_to(PROJECT_ROOT)}:{lineno}: {line.strip()}")
    return hits


def bullet_list(items: list[str]) -> str:
    if not items:
        return "- _None found_"
    return "\n".join(f"- `{item}`" for item in items)


tables = {name: load_csv(relative_path) for name, relative_path in artifact_paths.items()}
missing_artifacts = [
    relative_path for name, relative_path in artifact_paths.items() if tables[name] is None
]

feature_config = json.loads(read_text("config/ch1_feature_sets.json"))
run_config = json.loads(read_text("config/ch1_run_config.json"))

primary_model_ready = tables["primary_model_ready"]
extended_model_ready = tables["extended_model_ready"]
candidate_instances = tables["candidate_instances"]
valid_instances = tables["valid_instances"]
proxy_labels = tables["proxy_labels"]
usable_proxy_labels = tables["usable_proxy_labels"]
proxy_label_summary_by_horizon = tables["proxy_label_summary_by_horizon"]
proxy_unlabeled_reason_summary = tables["proxy_unlabeled_reason_summary"]
feature_set_definition = tables["feature_set_definition"]
feature_set_validation_summary = tables["feature_set_validation_summary"]
primary_readiness_summary = tables["primary_readiness_summary"]
primary_carry_forward_verification = tables["primary_carry_forward_verification"]
primary_missingness_by_hospital_and_family = tables["primary_missingness_by_hospital_and_family"]
retained_stay_table = tables["retained_stay_table"]
retained_hospitals = tables["retained_hospitals"]
site_counts_summary = tables["site_counts_summary"]
cohort_notes = tables["cohort_notes"]
cohort_verification_summary = tables["cohort_verification_summary"]
stay_split_assignments = tables["stay_split_assignments"]
stay_split_summary = tables["stay_split_summary"]
stay_split_verification_summary = tables["stay_split_verification_summary"]
primary_split_summary = tables["primary_split_summary"]
primary_split_verification_summary = tables["primary_split_verification_summary"]

dataset_key_columns = ["stay_id_global", "hospital_id", "block_index", "prediction_time_h"]
dataset_exists = (
    primary_model_ready is not None
    and extended_model_ready is not None
    and has_columns(primary_model_ready, dataset_key_columns)
    and primary_model_ready["instance_id"].nunique() == len(primary_model_ready)
)
dataset_status = "MET" if dataset_exists else ("PARTIALLY MET" if primary_model_ready is not None else "NOT MET")

candidate_has_valid_flag = has_columns(candidate_instances, ["valid_instance", "exclusion_reason"])
valid_instances_excluded_upstream = (
    valid_instances is not None
    and not valid_instances.empty
    and candidate_has_valid_flag
    and as_bool_series(valid_instances["valid_instance"]).all()
)
valid_instance_status = "MET" if candidate_has_valid_flag and valid_instances_excluded_upstream else "NOT MET"

label_columns = [
    "horizon_h",
    "label_value",
    "proxy_horizon_labelable",
    "unlabeled_reason",
    "label_definition_id",
    "label_definition_status",
    "event_time_proxy_h",
]
label_horizons = (
    sorted(pd.to_numeric(proxy_labels["horizon_h"], errors="coerce").dropna().astype(int).unique().tolist())
    if has_columns(proxy_labels, ["horizon_h"])
    else []
)
label_definition_ok = (
    proxy_labels is not None
    and not proxy_labels.empty
    and proxy_labels["label_definition_id"].astype(str).eq("proxy_within_horizon_icu_mortality_v1").all()
)
label_status = (
    "MET"
    if has_columns(proxy_labels, label_columns) and label_horizons == REQUIRED_HORIZONS and label_definition_ok
    else ("PARTIALLY MET" if proxy_labels is not None else "NOT MET")
)

primary_features = feature_config.get("primary_features", [])
extended_additional_features = feature_config.get("extended_additional_features", [])
feature_overlap = sorted(set(primary_features) & set(extended_additional_features))
feature_validation_primary = (
    feature_set_validation_summary.loc[
        feature_set_validation_summary["feature_set_name"].eq("primary")
    ].iloc[0]
    if feature_set_validation_summary is not None
    and not feature_set_validation_summary.empty
    and feature_set_validation_summary["feature_set_name"].eq("primary").any()
    else None
)
primary_feature_set_clear = (
    bool(primary_features)
    and not feature_overlap
    and feature_set_definition is not None
    and feature_set_definition["feature_set_name"].astype(str).isin(["primary", "extended"]).all()
    and feature_validation_primary is not None
    and int(feature_validation_primary["missing_from_blocked_schema_count"]) == 0
    and int(feature_validation_primary["absent_from_retained_data_count"]) == 0
)
feature_set_status = "MET" if primary_feature_set_clear else "NOT MET"

missingness_indicator_columns = [
    column for column in (primary_model_ready.columns if primary_model_ready is not None else [])
    if column.endswith("_missing_after_locf")
]
observed_indicator_columns = [
    column for column in (primary_model_ready.columns if primary_model_ready is not None else [])
    if column.endswith("_observed_in_block")
]
remaining_missing_cells = (
    int(primary_model_ready.isna().sum().sum()) if primary_model_ready is not None else 0
)
global_imputation_applied = as_bool_value(
    metric_value(primary_readiness_summary, "global_median_imputation_applied_in_export")
)
downstream_imputation_policy = metric_value(
    primary_readiness_summary,
    "downstream_imputation_policy",
)
locf_checks_ok = (
    check_value(primary_carry_forward_verification, "no_global_median_imputation_applied_in_export") is True
    and check_value(primary_carry_forward_verification, "no_ventilator_locf_outside_supported_windows") is True
)
model_time_imputation_hits = search_repo(
    r"SimpleImputer|KNNImputer|IterativeImputer|sklearn\\.impute",
    ["src", "tests", "pyproject.toml"],
)
missingness_status = (
    "MET"
    if primary_model_ready is not None
    and bool(missingness_indicator_columns)
    and bool(observed_indicator_columns)
    and remaining_missing_cells > 0
    and global_imputation_applied is False
    and downstream_imputation_policy == "deferred_to_model_training_stage"
    and locf_checks_ok
    and not model_time_imputation_hits
    else ("PARTIALLY MET" if primary_model_ready is not None else "NOT MET")
)

all_retained_stays_assigned = (
    check_value(stay_split_verification_summary, "all_retained_stays_assigned_to_split") is True
)
single_split_per_stay = (
    check_value(stay_split_verification_summary, "no_stay_in_more_than_one_split") is True
    and check_value(primary_split_verification_summary, "all_instances_from_stay_share_same_split") is True
)
site_representation_preserved = (
    stay_split_summary is not None
    and not stay_split_summary.empty
    and stay_split_summary.loc[stay_split_summary["summary_level"].eq("hospital"), "stay_count"].gt(0).all()
)
test_positive_stays = (
    int(
        stay_split_summary.loc[
            stay_split_summary["summary_level"].eq("overall")
            & stay_split_summary["split"].eq("test"),
            "positive_stays",
        ].iloc[0]
    )
    if stay_split_summary is not None
    and not stay_split_summary.empty
    and stay_split_summary["summary_level"].eq("overall").any()
    else 0
)
test_positive_instances = (
    int(
        primary_split_summary.loc[
            primary_split_summary["summary_level"].eq("overall")
            & primary_split_summary["split"].eq("test"),
            "positive_labels",
        ].iloc[0]
    )
    if primary_split_summary is not None
    and not primary_split_summary.empty
    and primary_split_summary["summary_level"].eq("overall").any()
    else 0
)
split_status = (
    "MET"
    if all_retained_stays_assigned and single_split_per_stay and site_representation_preserved and test_positive_instances > 0
    else (
        "PARTIALLY MET"
        if all_retained_stays_assigned and single_split_per_stay and site_representation_preserved
        else "NOT MET"
    )
)

baseline_model_code_hits = search_repo(
    r"LogisticRegression|xgboost|XGBClassifier|predict_proba|roc_auc_score|average_precision_score|calibration_curve|joblib|pickle",
    ["src", "tests", "pyproject.toml"],
)
reusable_preprocessing_modules = [
    "src/chapter1_mortality_decomposition/model_ready.py",
    "src/chapter1_mortality_decomposition/splits.py",
    "src/chapter1_mortality_decomposition/pipeline.py",
    "src/chapter1_mortality_decomposition/cli.py",
]
baseline_scaffold_status = "PARTIALLY MET" if not baseline_model_code_hits else "MET"

artifact_directories = (
    sorted(path.name for path in ARTIFACT_ROOT.iterdir() if path.is_dir())
    if ARTIFACT_ROOT.exists()
    else []
)
expected_preprocessing_directories = [
    "carry_forward",
    "cohort",
    "feature_sets",
    "instances",
    "labels",
    "model_ready",
    "observation_process",
    "splits",
]
modeling_directories = [
    directory for directory in artifact_directories
    if directory in {"baseline_models", "predictions", "evaluation", "metrics", "plots"}
]
artifact_structure_status = (
    "PARTIALLY MET"
    if set(expected_preprocessing_directories).issubset(set(artifact_directories))
    else "NOT MET"
)

checklist_rows = [
    {
        "item": "model-ready ASIC dataset exists",
        "status": dataset_status,
        "detail": "Primary and extended model-ready tables exist with prediction-instance keys.",
    },
    {
        "item": "valid prediction instances are encoded",
        "status": valid_instance_status,
        "detail": "Candidate rows carry validity flags and the valid table retains only valid instances.",
    },
    {
        "item": "frozen horizon labels exist",
        "status": label_status,
        "detail": "Proxy within-horizon labels are present for 8h, 16h, 24h, 48h, and 72h.",
    },
    {
        "item": "primary feature set is frozen and identifiable",
        "status": feature_set_status,
        "detail": "Primary vs extended feature definitions are versioned in config and materialized in artifacts.",
    },
    {
        "item": "split objects exist and are reusable",
        "status": split_status,
        "detail": "Deterministic stay-level split assignments exist, but current class balance must still be checked.",
    },
    {
        "item": "downstream missingness boundary is clear",
        "status": missingness_status,
        "detail": "Bounded LOCF happens upstream; final imputation remains deferred to training only.",
    },
    {
        "item": "baseline-model scaffold exists or can be added cleanly",
        "status": baseline_scaffold_status,
        "detail": "No model-training code exists yet, but the preprocessing interface is clean and reusable.",
    },
    {
        "item": "output/artifact structure is clear",
        "status": artifact_structure_status,
        "detail": "Preprocessing output structure is standardized, but no modeling output structure exists yet.",
    },
]

blockers: list[str] = []
nice_to_have_cleanup: list[str] = []

if missing_artifacts:
    blockers.append(
        "Some expected Chapter 1 artifacts are missing: " + ", ".join(missing_artifacts)
    )

if test_positive_instances == 0:
    blockers.append(
        "The canonical test split currently has zero positive labels in the primary model-ready table. "
        "That blocks meaningful frozen test-set AUROC, AUPRC, and calibration evaluation."
    )

if not baseline_model_code_hits:
    nice_to_have_cleanup.append(
        "There is no existing logistic-regression or XGBoost training / evaluation scaffold yet."
    )

if not modeling_directories:
    nice_to_have_cleanup.append(
        "Preprocessing outputs are standardized, but there is no model / prediction / metrics directory convention yet."
    )

if artifact_structure_status == "PARTIALLY MET":
    nice_to_have_cleanup.append(
        "Future ASIC and MIMIC modeling outputs should mirror a shared artifact layout once Sprint 2 implementation begins."
    )

if any(row["status"] == "NOT MET" for row in checklist_rows):
    overall_status = "Not ready"
elif blockers:
    overall_status = "Partially ready"
else:
    overall_status = "Ready"

can_safely_proceed = not blockers and not any(row["status"] == "NOT MET" for row in checklist_rows)

display(Markdown("## Executive Summary"))
display(
    Markdown(
        f"""
**Status:** {overall_status}

The Chapter 1 preprocessing layer already provides retained stays, valid prediction instances, proxy within-horizon labels, feature-set-specific model-ready tables, carry-forward QC, and deterministic stay-level split objects. The main blocker is current split balance rather than preprocessing availability: the canonical test split presently has **{test_positive_stays} positive stays** and **{test_positive_instances} positive primary model-ready labels**, so frozen baseline evaluation cannot yet proceed safely on test as-is. No baseline modeling code exists yet, but the preprocessing interface is clean enough to add one without refactoring.
"""
    )
)

display(Markdown("## Preconditions Checklist"))
display(pd.DataFrame(checklist_rows))

candidate_invalid_count = (
    int((~as_bool_series(candidate_instances["valid_instance"])).sum())
    if candidate_has_valid_flag
    else None
)
nonzero_unlabeled = (
    proxy_unlabeled_reason_summary.loc[
        pd.to_numeric(proxy_unlabeled_reason_summary["instance_count"], errors="coerce").fillna(0).gt(0)
    ]
    if proxy_unlabeled_reason_summary is not None
    else None
)

evidence_rows = [
    {
        "item": "model-ready ASIC dataset exists",
        "status": dataset_status,
        "paths": [
            artifact_paths["primary_model_ready"],
            artifact_paths["extended_model_ready"],
            "src/chapter1_mortality_decomposition/model_ready.py",
            "src/chapter1_mortality_decomposition/pipeline.py",
        ],
        "functions": [
            "build_chapter1_model_ready_dataset",
            "build_chapter1_dataset",
        ],
        "columns": dataset_key_columns + ["instance_id", "horizon_h", "label_value", "split"],
        "observation": (
            f"Primary rows: {len(primary_model_ready) if primary_model_ready is not None else 0}; "
            f"extended rows: {len(extended_model_ready) if extended_model_ready is not None else 0}. "
            "Rows are unique prediction instances keyed by instance_id."
        ),
    },
    {
        "item": "valid prediction instances are encoded",
        "status": valid_instance_status,
        "paths": [
            artifact_paths["candidate_instances"],
            artifact_paths["valid_instances"],
            "src/chapter1_mortality_decomposition/instances.py",
        ],
        "functions": ["build_chapter1_valid_instances"],
        "columns": ["valid_instance", "exclusion_reason", "covered_core_vital_group_count", "prediction_time_h", "horizon_h"],
        "observation": (
            f"Candidate instances carry validity flags; invalid candidate rows: {candidate_invalid_count}. "
            "The exported valid-instance table contains only rows where valid_instance is true."
        ),
    },
    {
        "item": "frozen horizon labels exist",
        "status": label_status,
        "paths": [
            artifact_paths["proxy_labels"],
            artifact_paths["usable_proxy_labels"],
            artifact_paths["proxy_label_summary_by_horizon"],
            artifact_paths["proxy_unlabeled_reason_summary"],
            "src/chapter1_mortality_decomposition/labels.py",
            "docs/label_logic_audit.md",
        ],
        "functions": ["build_chapter1_proxy_horizon_labels"],
        "columns": label_columns,
        "observation": (
            f"Horizon values found: {label_horizons}. Labels are long-form via horizon_h plus label_value, not wide one-column-per-horizon labels. "
            f"Usable label rows: {len(usable_proxy_labels) if usable_proxy_labels is not None else 0}."
        ),
    },
    {
        "item": "primary feature set is frozen and identifiable",
        "status": feature_set_status,
        "paths": [
            "config/ch1_feature_sets.json",
            artifact_paths["feature_set_definition"],
            artifact_paths["feature_set_validation_summary"],
            "src/chapter1_mortality_decomposition/config.py",
        ],
        "functions": [
            "load_chapter1_feature_set_config",
            "build_chapter1_feature_set_definition",
        ],
        "columns": [
            "feature_set_name",
            "feature_source_group",
            "feature_name",
            "base_variable",
            "selected_for_model",
        ],
        "observation": (
            f"Primary base features: {len(primary_features)}; extended additional base features: {len(extended_additional_features)}; overlap: {feature_overlap}. "
            "Primary and extended are programmatically selectable and versioned."
        ),
    },
    {
        "item": "split objects exist and are reusable",
        "status": split_status,
        "paths": [
            artifact_paths["stay_split_assignments"],
            artifact_paths["stay_split_summary"],
            artifact_paths["stay_split_verification_summary"],
            artifact_paths["primary_split_summary"],
            artifact_paths["primary_split_verification_summary"],
            "src/chapter1_mortality_decomposition/splits.py",
            "config/ch1_run_config.json",
        ],
        "functions": ["build_chapter1_stay_splits"],
        "columns": ["stay_id_global", "hospital_id", "icu_mortality", "split", "split_random_seed"],
        "observation": (
            f"Split seed: {run_config.get('split_random_seed')}. All retained stays are assigned exactly once, and all rows from a stay share one split. "
            f"Current blocker: test positive stays = {test_positive_stays}, test positive labels = {test_positive_instances}."
        ),
    },
    {
        "item": "downstream missingness boundary is clear",
        "status": missingness_status,
        "paths": [
            artifact_paths["primary_model_ready"],
            artifact_paths["primary_readiness_summary"],
            artifact_paths["primary_carry_forward_verification"],
            artifact_paths["primary_missingness_by_hospital_and_family"],
            "src/chapter1_mortality_decomposition/carry_forward.py",
            "src/chapter1_mortality_decomposition/model_ready.py",
        ],
        "functions": [
            "build_chapter1_locf_feature_frame",
            "build_chapter1_model_ready_dataset",
        ],
        "columns": [
            "*_observed_in_block",
            "*_filled_by_locf",
            "*_missing_after_locf",
            "split",
        ],
        "observation": (
            f"Missingness indicator columns found: {len(missingness_indicator_columns)}; remaining missing cells in primary model-ready table: {remaining_missing_cells}. "
            f"Downstream imputation policy: {downstream_imputation_policy}."
        ),
    },
    {
        "item": "baseline-model scaffold exists or can be added cleanly",
        "status": baseline_scaffold_status,
        "paths": reusable_preprocessing_modules,
        "functions": ["clean reusable preprocessing interface already exists"],
        "columns": [],
        "observation": (
            "No logistic-regression / XGBoost training or evaluation code is currently present in src/tests. "
            "The later modeling layer can be added cleanly on top of the exported model-ready tables."
        ),
    },
    {
        "item": "output/artifact structure is clear",
        "status": artifact_structure_status,
        "paths": ["artifacts/chapter1", "src/chapter1_mortality_decomposition/pipeline.py", "README.md"],
        "functions": ["write_chapter1_dataset"],
        "columns": [],
        "observation": (
            f"Preprocessing directories found: {artifact_directories}. "
            f"Modeling-specific directories found: {modeling_directories or ['none']}."
        ),
    },
]

display(Markdown("## Evidence by Precondition"))
for evidence in evidence_rows:
    display(
        Markdown(
            f"""
### {evidence['item']} [{evidence['status']}]

**Relevant paths**
{bullet_list(evidence['paths'])}

**Relevant functions / modules**
{bullet_list(evidence['functions'])}

**Exact columns**
{bullet_list(evidence['columns'])}

**Observation**
{evidence['observation']}
"""
        )
    )

display(Markdown("## Gaps and Blockers"))
if blockers:
    display(Markdown("\n".join(f"- {blocker}" for blocker in blockers)))
else:
    display(Markdown("- No blocking gaps found."))

if nice_to_have_cleanup:
    display(Markdown("**Nice-to-have cleanup**"))
    display(Markdown("\n".join(f"- {item}" for item in nice_to_have_cleanup)))

display(Markdown("## Minimal Next Step Recommendation"))
if blockers:
    display(
        Markdown(
            "Regenerate or explicitly re-freeze the Chapter 1 split assignments so the test split contains at least one positive stay / label while keeping the current deterministic stay-level within-hospital split logic. After that, implement the thin baseline modeling layer on top of the primary model-ready table."
        )
    )
else:
    display(
        Markdown(
            "Proceed to implement logistic regression and XGBoost using the frozen primary model-ready dataset, with final imputation fit on the training split only."
        )
    )

display(Markdown("## Can We Safely Proceed to Logistic Regression and XGBoost Now?"))
display(Markdown(f"**{'Yes' if can_safely_proceed else 'No'}**"))

files_inspected_most_closely = [
    "README.md",
    "docs/chapter1_analysis_spec_frozen_v1.md",
    "docs/preprocessing_interface.md",
    "docs/label_logic_audit.md",
    "config/ch1_feature_sets.json",
    "config/ch1_run_config.json",
    "src/chapter1_mortality_decomposition/cohort.py",
    "src/chapter1_mortality_decomposition/instances.py",
    "src/chapter1_mortality_decomposition/labels.py",
    "src/chapter1_mortality_decomposition/carry_forward.py",
    "src/chapter1_mortality_decomposition/model_ready.py",
    "src/chapter1_mortality_decomposition/splits.py",
    "src/chapter1_mortality_decomposition/pipeline.py",
    artifact_paths["retained_stay_table"],
    artifact_paths["candidate_instances"],
    artifact_paths["valid_instances"],
    artifact_paths["proxy_labels"],
    artifact_paths["usable_proxy_labels"],
    artifact_paths["feature_set_definition"],
    artifact_paths["feature_set_validation_summary"],
    artifact_paths["primary_model_ready"],
    artifact_paths["extended_model_ready"],
    artifact_paths["stay_split_assignments"],
    artifact_paths["stay_split_summary"],
    artifact_paths["primary_split_summary"],
]
display(Markdown("## Files Inspected Most Closely"))
display(Markdown(bullet_list(files_inspected_most_closely)))


## Executive Summary


**Status:** Partially ready

The Chapter 1 preprocessing layer already provides retained stays, valid prediction instances, proxy within-horizon labels, feature-set-specific model-ready tables, carry-forward QC, and deterministic stay-level split objects. The main blocker is current split balance rather than preprocessing availability: the canonical test split presently has **0 positive stays** and **0 positive primary model-ready labels**, so frozen baseline evaluation cannot yet proceed safely on test as-is. No baseline modeling code exists yet, but the preprocessing interface is clean enough to add one without refactoring.


## Preconditions Checklist

,item,status,detail
0,model-ready ASIC dataset exists,MET,Primary and extended model-ready tables exist with prediction-instance keys.
1,valid prediction instances are encoded,MET,Candidate rows carry validity flags and the valid table retains only valid instances.
2,frozen horizon labels exist,MET,"Proxy within-horizon labels are present for 8h, 16h, 24h, 48h, and 72h."
3,primary feature set is frozen and identifiable,MET,Primary vs extended feature definitions are versioned in config and materialized in artifacts.
4,split objects exist and are reusable,PARTIALLY MET,"Deterministic stay-level split assignments exist, but current class balance must still be checked."
5,downstream missingness boundary is clear,MET,Bounded LOCF happens upstream; final imputation remains deferred to training only.
6,baseline-model scaffold exists or can be added cleanly,PARTIALLY MET,"No model-training code exists yet, but the preprocessing interface is clean and reusable."
7,output/artifact structure is clear,PARTIALLY MET,"Preprocessing output structure is standardized, but no modeling output structure exists yet."


## Evidence by Precondition

...

Regenerate or explicitly re-freeze the Chapter 1 split assignments so the test split contains at least one positive stay / label while keeping the current deterministic stay-level within-hospital split logic. After that, implement the thin baseline modeling layer on top of the primary model-ready table.

## Can We Safely Proceed to Logistic Regression and XGBoost Now?

**No**

## Files Inspected Most Closely

- `README.md`
- `docs/chapter1_analysis_spec_frozen_v1.md`
- `docs/preprocessing_interface.md`
- `docs/label_logic_audit.md`
- `config/ch1_feature_sets.json`
- `config/ch1_run_config.json`
- `src/chapter1_mortality_decomposition/cohort.py`
- `src/chapter1_mortality_decomposition/instances.py`
- `src/chapter1_mortality_decomposition/labels.py`
- `src/chapter1_mortality_decomposition/carry_forward.py`
- `src/chapter1_mortality_decomposition/model_ready.py`
- `src/chapter1_mortality_decomposition/splits.py`
- `src/chapter1_mortality_decomposition/pipeline.py`
- `artifacts/chapter1/cohort/chapter1_retained_stay_table.csv`
- `artifacts/chapter1/instances/chapter1_candidate_instances.csv`
- `artifacts/chapter1/instances/chapter1_valid_instances.csv`
- `artifacts/chapter1/labels/chapter1_proxy_horizon_labels.csv`
- `artifacts/chapter1/labels/chapter1_usable_proxy_horizon_labels.csv`
- `artifacts/chapter1/feature_sets/chapter1_feature_set_definition.csv`
- `artifacts/chapter1/feature_sets/chapter1_feature_set_validation_summary.csv`
- `artifacts/chapter1/model_ready/chapter1_primary_model_ready_dataset.csv`
- `artifacts/chapter1/model_ready/chapter1_extended_model_ready_dataset.csv`
- `artifacts/chapter1/splits/chapter1_stay_split_assignments.csv`
- `artifacts/chapter1/splits/chapter1_stay_split_summary.csv`
- `artifacts/chapter1/splits/chapter1_primary_split_summary.csv`

In [3]:
def maybe_display_table(title: str, df: pd.DataFrame | None) -> None:
    display(Markdown(f"### {title}"))
    if df is None:
        display(Markdown("_Missing artifact_"))
    else:
        display(df)


display(Markdown("## Supporting Tables"))
maybe_display_table("Label Summary by Horizon", proxy_label_summary_by_horizon)
maybe_display_table(
    "Unlabeled Reason Summary (Non-zero Rows)",
    nonzero_unlabeled if nonzero_unlabeled is not None else None,
)
maybe_display_table("Feature-set Validation Summary", feature_set_validation_summary)
maybe_display_table("Stay-level Split Summary", stay_split_summary)
maybe_display_table(
    "Primary Model-ready Split Summary (Overall Rows)",
    primary_split_summary.loc[primary_split_summary["summary_level"].eq("overall")]
    if primary_split_summary is not None
    else None,
)


## Supporting Tables

### Label Summary by Horizon

,horizon_h,total_valid_prediction_instances,labelable_instances,positive_labels,negative_labels,unlabeled_instances
0,8,1678,1256,10,1246,422
1,16,1678,1244,20,1224,434
2,24,1678,1232,30,1202,446
3,48,1678,1194,58,1136,484
4,72,1678,1154,82,1072,524


### Unlabeled Reason Summary (Non-zero Rows)

,horizon_h,unlabeled_reason,instance_count
1,8,prediction_time_not_strictly_before_proxy_end,1
2,8,survivor_without_full_horizon_observation,21
3,8,non_survivor_proxy_end_not_within_horizon,400
6,16,prediction_time_not_strictly_before_proxy_end,1
7,16,survivor_without_full_horizon_observation,43
8,16,non_survivor_proxy_end_not_within_horizon,390
11,24,prediction_time_not_strictly_before_proxy_end,1
12,24,survivor_without_full_horizon_observation,65
13,24,non_survivor_proxy_end_not_within_horizon,380
16,48,prediction_time_not_strictly_before_proxy_end,1


...

,feature_config_version,feature_set_name,primary_feature_count,extended_only_feature_count,total_extended_feature_count,configured_base_feature_count,available_in_blocked_schema_count,missing_from_blocked_schema_count,missing_from_blocked_schema_features,present_in_retained_data_count,absent_from_retained_data_count,absent_from_retained_data_features,duplicate_primary_feature_count,duplicate_extended_additional_feature_count,overlap_primary_extended_additional_count
0,ch1_feature_sets_v1,primary,31,15,46,31,31,0,[],31,0,[],0,0,0
1,ch1_feature_sets_v1,extended,31,15,46,46,46,0,[],46,0,[],0,0,0


### Stay-level Split Summary

,summary_level,hospital_id,split,stay_count,positive_stays,negative_stays,label_prevalence,actual_proportion,target_proportion
0,overall,NaN,train,25,8,17,0.320000,0.714286,0.70
1,overall,NaN,validation,6,2,4,0.333333,0.171429,0.15
2,overall,NaN,test,4,0,4,0.000000,0.114286,0.15
3,hospital,asic_UK02,train,7,3,4,0.428571,0.700000,0.70
4,hospital,asic_UK02,validation,2,1,1,0.500000,0.200000,0.15
5,hospital,asic_UK02,test,1,0,1,0.000000,0.100000,0.15
6,hospital,asic_UK04,train,7,2,5,0.285714,0.777778,0.70
7,hospital,asic_UK04,validation,1,0,1,0.000000,0.111111,0.15
8,hospital,asic_UK04,test,1,0,1,0.000000,0.111111,0.15
9,hospital,asic_UK07,train,6,2,4,0.333333,0.666667,0.70


### Primary Model-ready Split Summary (Overall Rows)

,feature_set_name,summary_level,split,hospital_id,horizon_h,stay_count,instance_count,positive_labels,negative_labels,label_prevalence
0,primary,overall,test,NaN,NaN,4,1111,0,1111,0.000000
6,primary,overall,train,NaN,NaN,25,3765,158,3607,0.041965
12,primary,overall,validation,NaN,NaN,6,1204,42,1162,0.034884
